# LangGraph example: a three-layer number pipeline

Type a number. One agent checks it is a whole number from 1 to 10, three agents each own a range, three more add facts, and a last one confirms the number. It is shaped like the incident-triage graph, with no Azure and no LLM.

**To run it:** Python 3.12 and `pip install langgraph`.

## The LangGraph pieces you will meet

Each cell below tags the pieces it uses, so you can spot them.

| Tag | What it is | Plain meaning | In this notebook |
|---|---|---|---|
| 🟦 **STATE** | A shared dict (`TypedDict`) every node can read | The whiteboard | `Pipeline` |
| 🟩 **NODE** | A plain function: reads state, returns only what it changes | One worker | `validate`, `low`, `mid`, ... |
| ➡️ **EDGE** | "After A, run B" | An arrow | `add_edge("reject", END)` |
| 🔀 **CONDITIONAL EDGE** | A function picks the next node(s) | A decision diamond | after `validate` |
| 🔱 **FAN-OUT** | One node leads to several, which run at the same time | Hand out jobs | layer 1 to layer 2 |
| 🔗 **FAN-IN** | A node with a list of sources waits for all of them | Wait for everyone | layer 2 to layer 3 |
| 🧮 **REDUCER** | A rule for merging parallel writes to one key | Merge the notes | `merge_findings`, `operator.add` |
| 🏁 **START / END** | The entry and exit of the graph | Door in, door out | `START`, `END` |

## The graph

```
 START
   |
   v
 LAYER 1   [ validate ] -- bad input --> [ reject ] --> END
   |
   | valid number: start all three at once
   v
 LAYER 2   [ low 1-3 ]     [ mid 4-7 ]     [ high 8-10 ]
                 \              |              /
                  \   wait until ALL THREE finish
                   \            |            /
                    v           v           v
 LAYER 3   [ even_odd ]      [ prime ]      [ size ]
                 \              |              /
                  \   wait until ALL THREE finish
                   \            |            /
                    v           v           v
 LAYER 4                   [ finalize ]
                                |
                                v
                               END
```

Read it top to bottom. Agents on the same row run at the same time. A new row starts only when every agent in the row above has finished.

| Layer | Nodes | Like in the project | LangGraph pieces |
|---|---|---|---|
| 1 | `validate`, `reject` | Triage supervisor | 🔀 conditional edge, 🔱 fan-out |
| 2 | `low`, `mid`, `high` | Logs, Metrics, Change agents | 🟩 nodes, 🧮 reducer, 🔗 fan-in |
| 3 | `even_odd`, `prime`, `size` | a second wave of specialists | 🔱 fan-out again, 🔗 fan-in |
| 4 | `finalize` | Postmortem | 🔗 fan-in, ➡️ edge to END |

The next cells build the graph one layer at a time. The nodes are plain functions, so in each cell we also **call them by hand** to see what they return. The graph itself is wired and run in the last two cells.

## Step 0: the state and the helpers

**What happens here:** we define the 🟦 **STATE**, the dict that travels through the whole graph. Every node reads from it and returns only the keys it wants to change.

Two keys need a 🧮 **REDUCER** because several nodes write to them at the same time:

| Key | Written by | Reducer | Effect |
|---|---|---|---|
| `findings` | every layer 2 and layer 3 node | `merge_findings` (merge two dicts) | all their facts end up in one dict |
| `trace` | every node | `operator.add` (join two lists) | one log line per node, in order |

Without a reducer, LangGraph refuses parallel writes to the same key.

`say()` is a small helper that builds one `trace` line, like `em.say` in the project.

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


def merge_findings(a: dict, b: dict) -> dict:       # 🧮 REDUCER: how to combine two writes
    return {**a, **b}


class Pipeline(TypedDict, total=False):             # 🟦 STATE: the shared whiteboard
    raw: str                                              # what the user typed
    number: int                                           # set by validate
    error: str                                            # set by validate when the input is bad
    findings: Annotated[dict, merge_findings]             # layers 2 and 3 all write here
    trace: Annotated[list, operator.add]                  # every node appends a line
    final: int                                            # set by finalize


def say(node: str, text: str) -> dict:
    return {"trace": [f"{node:9} {text}"]}


print("state keys:   ", list(Pipeline.__annotations__))
print("reducer demo: ", merge_findings({"low": False}, {"mid": True}))
print("say() output: ", say("validate", "hello"))

state keys:    ['raw', 'number', 'error', 'findings', 'trace', 'final']
reducer demo:  {'low': False, 'mid': True}
say() output:  {'trace': ['validate  hello']}


## Layer 1: validate (the supervisor)

**What happens here:** the first agent checks the input is a whole number from 1 to 10.

| Piece | Name | What it does |
|---|---|---|
| 🟩 **NODE** | `validate` | Reads `raw`. Returns `number` if valid, or `error` if not. Always adds a `trace` line. |
| 🟩 **NODE** | `reject` | Reads `error` and writes the final trace line. Only runs for bad input. |
| 🔀 **CONDITIONAL EDGE** (the router) | `route_after_validate` | Reads the state and returns where to go next. Bad input returns `"reject"`. Good input returns the **list** `["low", "mid", "high"]`, so all three start at once (🔱 **FAN-OUT**). |

Notice each node returns **only its own keys**. LangGraph merges them into the state.

The cell calls each function by hand with a sample state, to show what it returns.

In [2]:
LAYER2 = ["low", "mid", "high"]
LAYER3 = ["even_odd", "prime", "size"]


def validate(state: Pipeline) -> dict:                       # 🟩 NODE
    raw = state["raw"].strip()
    if not raw.lstrip("-").isdigit():
        return {"error": f"'{raw}' is not a whole number", **say("validate", "rejected: not a number")}
    n = int(raw)
    if not 1 <= n <= 10:
        return {"error": f"{n} is outside 1-10", **say("validate", "rejected: out of range")}
    return {"number": n, **say("validate", f"{n} is valid, sending to the range agents")}


def route_after_validate(state: Pipeline):                   # 🔀 CONDITIONAL EDGE (router)
    return "reject" if state.get("error") else LAYER2        # a list = run all of these in parallel


def reject(state: Pipeline) -> dict:                         # 🟩 NODE
    return say("reject", state["error"])


for text in ("6", "11", "abc"):
    out = validate({"raw": text})
    print(f"validate({text!r:6}) -> {out}")
    print(f"   router sends it to: {route_after_validate(out)}\n")

validate('6'   ) -> {'number': 6, 'trace': ['validate  6 is valid, sending to the range agents']}
   router sends it to: ['low', 'mid', 'high']

validate('11'  ) -> {'error': '11 is outside 1-10', 'trace': ['validate  rejected: out of range']}
   router sends it to: reject

validate('abc' ) -> {'error': "'abc' is not a whole number", 'trace': ['validate  rejected: not a number']}
   router sends it to: reject



## Layer 2: the three range agents

**What happens here:** three agents each own a range (1-3, 4-7, 8-10). All three run **at the same time** and each says whether the number is in its range.

| Piece | Name | What it does |
|---|---|---|
| 🟩 **NODE** x3 | `low`, `mid`, `high` | Read `number`. Return `{"findings": {name: True or False}}` plus a trace line. |
| 🧮 **REDUCER** | `merge_findings` | The three agents write to `findings` together. The reducer merges their three small dicts into one. |

The three nodes come from one factory, `range_agent(name, low, high)`, so the code is written once.

Below, we call each by hand with `number = 6`. Only `mid` should claim it.

In [3]:
def range_agent(name: str, low: int, high: int):         # a factory that builds one 🟩 NODE
    def node(state: Pipeline) -> dict:
        mine = low <= state["number"] <= high
        text = "my range, claiming it" if mine else "not mine"
        return {"findings": {name: mine}, **say(name, f"{low}-{high}: {text}")}
    node.__name__ = name
    return node


low_node = range_agent("low", 1, 3)
mid_node = range_agent("mid", 4, 7)
high_node = range_agent("high", 8, 10)

state = {"number": 6}
outputs = [low_node(state), mid_node(state), high_node(state)]
for out in outputs:
    print(out)

merged = {}
for out in outputs:
    merged = merge_findings(merged, out["findings"])       # what the 🧮 REDUCER does for us
print("\nafter the reducer merges the three:", merged)

{'findings': {'low': False}, 'trace': ['low       1-3: not mine']}
{'findings': {'mid': True}, 'trace': ['mid       4-7: my range, claiming it']}
{'findings': {'high': False}, 'trace': ['high      8-10: not mine']}

after the reducer merges the three: {'low': False, 'mid': True, 'high': False}


## Layer 3: the three fact agents

**What happens here:** after **all** of layer 2 has finished, three more agents run at the same time. Each adds one fact about the number.

| Piece | Name | What it adds to `findings` |
|---|---|---|
| 🟩 **NODE** | `even_odd` | `parity`: even or odd |
| 🟩 **NODE** | `prime` | `prime`: True or False |
| 🟩 **NODE** | `size` | `size`: small (1-3), medium (4-7) or large (8-10) |

In the graph, every layer 2 node connects to every layer 3 node, and each layer 3 node waits for all three (🔗 **FAN-IN**, then 🔱 **FAN-OUT** again). We add those edges two cells down.

Below, we call each by hand with `number = 6`.

In [4]:
def even_odd(state: Pipeline) -> dict:                   # 🟩 NODE
    kind = "even" if state["number"] % 2 == 0 else "odd"
    return {"findings": {"parity": kind}, **say("even_odd", kind)}


def prime(state: Pipeline) -> dict:                      # 🟩 NODE
    n = state["number"]
    is_prime = n > 1 and all(n % d for d in range(2, n))
    return {"findings": {"prime": is_prime}, **say("prime", "prime" if is_prime else "not prime")}


def size(state: Pipeline) -> dict:                       # 🟩 NODE
    n = state["number"]
    label = "small" if n <= 3 else "medium" if n <= 7 else "large"
    return {"findings": {"size": label}, **say("size", label)}


state = {"number": 6}
for fn in (even_odd, prime, size):
    print(fn(state))

{'findings': {'parity': 'even'}, 'trace': ['even_odd  even']}
{'findings': {'prime': False}, 'trace': ['prime     not prime']}
{'findings': {'size': 'medium'}, 'trace': ['size      medium']}


## Layer 4: finalize

**What happens here:** the last agent waits for all of layer 3, then confirms the number and puts it in `final`.

| Piece | Name | What it does |
|---|---|---|
| 🟩 **NODE** | `finalize` | Reads `number` and `findings`. Finds which range agent claimed the number, writes `final`, adds a trace line. |
| ➡️ **EDGE** | `finalize -> END` | Added in the wiring cell. After `finalize`, the graph stops. |

Below, we call it by hand with a state that looks like what layers 1 to 3 would have built.

In [5]:
def finalize(state: Pipeline) -> dict:                   # 🟩 NODE
    owner = next(k for k in ("low", "mid", "high") if state["findings"][k])
    return {"final": state["number"], **say("finalize", f"{state['number']} confirmed by '{owner}'")}


state = {
    "number": 6,
    "findings": {"low": False, "mid": True, "high": False, "parity": "even", "prime": False, "size": "medium"},
}
print(finalize(state))

{'final': 6, 'trace': ["finalize  6 confirmed by 'mid'"]}


## Wiring: nodes and edges into a graph

**What happens here:** until now we only had functions. This cell turns them into a LangGraph.

| Call | Piece | Meaning |
|---|---|---|
| `StateGraph(Pipeline)` | 🟦 STATE | "A graph whose state looks like `Pipeline`" |
| `add_node(name, fn)` | 🟩 NODE | Register a function under a name |
| `add_edge(START, "validate")` | ➡️ EDGE + 🏁 START | The first node to run |
| `add_conditional_edges("validate", route_after_validate, [...])` | 🔀 CONDITIONAL EDGE | After `validate`, ask the router where to go. The list names every place it may send us. |
| `add_edge(LAYER2, agent)` | 🔗 FAN-IN | A **list** as the source means: wait until all of them finish |
| `add_edge(LAYER3, "finalize")` | 🔗 FAN-IN | `finalize` waits for all of layer 3 |
| `add_edge("finalize", END)`, `add_edge("reject", END)` | ➡️ EDGE + 🏁 END | The two ways the graph can stop |
| `compile()` | | Check the wiring and make it runnable |

The cell then prints every edge LangGraph recorded. Dotted means conditional.

In [6]:
g = StateGraph(Pipeline)                                     # 🟦 STATE

g.add_node("validate", validate)                             # 🟩 NODES
g.add_node("reject", reject)
g.add_node("low", low_node)
g.add_node("mid", mid_node)
g.add_node("high", high_node)
for fn in (even_odd, prime, size, finalize):
    g.add_node(fn.__name__, fn)

g.add_edge(START, "validate")                                # ➡️ EDGE from 🏁 START
g.add_conditional_edges("validate", route_after_validate, ["reject", *LAYER2])   # 🔀 + 🔱
g.add_edge("reject", END)                                    # ➡️ EDGE to 🏁 END
for agent in LAYER3:
    g.add_edge(LAYER2, agent)                                # 🔗 FAN-IN, then 🔱 FAN-OUT
g.add_edge(LAYER3, "finalize")                               # 🔗 FAN-IN
g.add_edge("finalize", END)                                  # ➡️ EDGE to 🏁 END

pipeline = g.compile()

for edge in pipeline.get_graph().edges:
    arrow = "..>" if edge.conditional else "-->"
    print(f"{edge.source:10} {arrow} {edge.target}")

__start__  --> validate
even_odd   --> finalize
high       --> even_odd
high       --> prime
high       --> size
low        --> even_odd
low        --> prime
low        --> size
mid        --> even_odd
mid        --> prime
mid        --> size
prime      --> finalize
size       --> finalize
validate   ..> high
validate   ..> low
validate   ..> mid
validate   ..> reject
finalize   --> __end__
reject     --> __end__


## Run the whole graph

**What happens here:** we give the graph only `{"raw": "6"}` and LangGraph runs everything in order: layer 1, then layer 2 in parallel, then layer 3 in parallel, then layer 4.

The printed `trace` is the log each node added to the state. The order **inside** a layer can change between runs, because parallel nodes have no fixed order.

Four inputs, to see every path:

| Input | Expected path |
|---|---|
| `6` | all four layers; `mid` claims it |
| `2` | all four layers; `low` claims it |
| `11` | stops at `reject`; layers 2 to 4 never run |
| `abc` | stops at `reject`; layers 2 to 4 never run |

In [7]:
def run(text: str):
    out = pipeline.invoke({"raw": text})                     # the whole graph, start to END
    print(f"input {text!r}")
    for line in out["trace"]:
        print("   ", line)
    print("    final:", out.get("final"), "| error:", out.get("error"))
    print("    findings:", out.get("findings"), "\n")


run("6")
run("2")
run("11")
run("abc")

input '6'
    validate  6 is valid, sending to the range agents
    high      8-10: not mine
    low       1-3: not mine
    mid       4-7: my range, claiming it
    even_odd  even
    prime     not prime
    size      medium
    finalize  6 confirmed by 'mid'
    final: 6 | error: None
    findings: {'high': False, 'low': False, 'mid': True, 'parity': 'even', 'prime': False, 'size': 'medium'} 

input '2'
    validate  2 is valid, sending to the range agents
    high      8-10: not mine
    low       1-3: my range, claiming it
    mid       4-7: not mine
    even_odd  even
    prime     prime
    size      small
    finalize  2 confirmed by 'low'
    final: 2 | error: None
    findings: {'high': False, 'low': True, 'mid': False, 'parity': 'even', 'prime': True, 'size': 'small'} 

input '11'
    validate  rejected: out of range
    reject    11 is outside 1-10
    final: None | error: 11 is outside 1-10
    findings: {} 

input 'abc'
    validate  rejected: not a number
    reject    'a

## What to notice

| Observation | Why |
|---|---|
| `validate` always prints first | it is the only node after 🏁 `START` |
| `low`, `mid`, `high` all run, even when two say "not mine" | 🔱 fan-out runs every branch; each decides for itself |
| Layer 3 starts only after all three range agents | 🔗 a list as the source of `add_edge` means "wait for all" |
| `finalize` prints last | it waits for all of layer 3 |
| Bad input skips layers 2 to 4 | the 🔀 conditional edge sent it to `reject` |
| `findings` holds keys from six different nodes | the 🧮 reducer merged the parallel writes |
| Each node returned only a few keys, yet `final` appears in the result | LangGraph merges every node's return into the 🟦 state |

## Where to look next in the project

| Concept | File |
|---|---|
| State, reducer, nodes, edges, parallel, conditional edges | `backend/src/triage/graph.py` |
| Agent pattern (read, announce, act, report) | `backend/src/triage/agents.py` |